# Stage 2A: Baseline CNN (from scratch)
Textbook mapping: **Data Mining** step of the processing chain.

A small CNN trained from scratch. It is the **baseline**: the simplest reasonable model. Notebook 03 (transfer learning) must beat it, and the comparison goes in your Results section.

**Before running**
- Runtime > Change runtime type > **T4 GPU**
- The shared folder `road_project/outputs/` must contain `processed_dataset.zip` and `preprocessing_config.json` (made by notebook 01)
- Only training and **validation** data are used here. The test set stays untouched until Stage 3.

## 0. Setup

In [ ]:
import os, json, zipfile, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/road_project')
OUT_DIR    = DRIVE_ROOT / 'outputs'
MODEL_DIR  = DRIVE_ROOT / 'models'
FIG_DIR    = OUT_DIR / 'figures'
for d in (MODEL_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42                       # must match Stage 1. Do not change.
keras.utils.set_random_seed(SEED)
sns.set_theme(style='whitegrid')

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__, '| GPUs:', gpus)
if not gpus:
    print('WARNING: no GPU. Runtime > Change runtime type > T4 GPU, then rerun.')

## 1. Settings (change these, then bump the version in `RUN_NAME`)

In [ ]:
RUN_NAME      = 'baseline_cnn_v1'     # new name for every experiment: baseline_cnn_v2, ...
EPOCHS        = 40                    # early stopping will usually end sooner
BATCH_SIZE    = 32
LEARNING_RATE = 1e-3
DROPOUT       = 0.5
FILTERS       = (32, 64, 128, 256)    # one conv block per entry

## 2. Load data

In [ ]:
# Unzip the shared processed dataset to fast local disk (skipped if already done)
PROC = Path('/content/processed')
if not PROC.exists():
    with zipfile.ZipFile(OUT_DIR / 'processed_dataset.zip') as z:
        z.extractall(PROC)

cfg = json.load(open(OUT_DIR / 'preprocessing_config.json'))
assert cfg['seed'] == SEED, 'Config seed differs from SEED. Use the shared dataset.'
CLASS_NAMES   = cfg['class_names']
IMG_SIZE      = cfg['img_size']
CLASS_WEIGHTS = {int(k): v for k, v in cfg['class_weights'].items()}   # JSON keys are strings

def load(split, shuffle):
    return keras.utils.image_dataset_from_directory(
        PROC / split, class_names=CLASS_NAMES, image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE, label_mode='int', shuffle=shuffle, seed=SEED)

# NOTE: the test set is deliberately NOT loaded here. It is used once, in Stage 3.
train_ds = load('train', True).prefetch(tf.data.AUTOTUNE)
val_ds   = load('val', False).prefetch(tf.data.AUTOTUNE)

counts = {s: {c: len(list((PROC / s / c).glob('*.jpg'))) for c in CLASS_NAMES} for s in ('train', 'val')}
print(pd.DataFrame(counts))
print('\nClass weights:', {CLASS_NAMES[k]: round(v, 2) for k, v in CLASS_WEIGHTS.items()})

## 3. Build the model

In [ ]:
# Augmentation lives inside the model, so it is active during training only
# (Keras switches these layers off automatically for validation, testing and prediction).
augment = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
    layers.RandomContrast(0.2),
], name='augment')

In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return layers.MaxPooling2D()(x)

def build_baseline():
    inputs = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = augment(inputs)                       # training-only augmentation
    x = layers.Rescaling(1 / 255)(x)          # normalize pixels 0-255 -> 0-1
    for f in FILTERS:
        x = conv_block(x, f)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(DROPOUT)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(DROPOUT / 2)(x)
    outputs = layers.Dense(len(CLASS_NAMES), activation='softmax')(x)
    return keras.Model(inputs, outputs, name=RUN_NAME)

model = build_baseline()
model.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 4. Train
- **Early stopping** stops when validation loss stops improving (prevents overfitting)
- **Checkpoint** keeps the best model on Drive
- **ReduceLROnPlateau** lowers the learning rate when progress stalls
- **class_weight** makes rare classes count more in the loss

In [ ]:
CKPT = MODEL_DIR / f'{RUN_NAME}.keras'
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(CKPT, monitor='val_loss', save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                    class_weight=CLASS_WEIGHTS, callbacks=callbacks)
model = keras.models.load_model(CKPT)          # reload the best checkpoint

## 5. Learning curves

In [ ]:
def plot_history(h, path, vline=None):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for a, key, title in zip(ax, ('accuracy', 'loss'), ('Accuracy', 'Loss')):
        a.plot(h[key], label='train'); a.plot(h['val_' + key], label='validation')
        if vline is not None: a.axvline(vline - 0.5, color='gray', ls='--', label='fine-tuning starts')
        a.set_title(title); a.set_xlabel('epoch'); a.legend()
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

def evaluate_split(model, ds, split_name, fig_path):
    y_true = np.concatenate([y.numpy() for _, y in ds])
    y_pred = model.predict(ds, verbose=0).argmax(1)
    labels = list(range(len(CLASS_NAMES)))
    text = classification_report(y_true, y_pred, labels=labels, target_names=CLASS_NAMES, zero_division=0)
    rep = classification_report(y_true, y_pred, labels=labels, target_names=CLASS_NAMES,
                                zero_division=0, output_dict=True)
    print(f'--- {split_name} ---'); print(text)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel('predicted'); plt.ylabel('true'); plt.title(f'{RUN_NAME}: {split_name} confusion matrix (row-normalized)')
    plt.tight_layout(); plt.savefig(fig_path, dpi=150); plt.show()
    return rep, cm

def save_run(history_dict, rep, extra):
    json.dump(history_dict, open(OUT_DIR / f'history_{RUN_NAME}.json', 'w'))
    metrics = {'run': RUN_NAME, 'val_accuracy': rep['accuracy'],
               'val_macro_f1': rep['macro avg']['f1-score'],
               'per_class_f1': {c: rep[c]['f1-score'] for c in CLASS_NAMES}, **extra}
    json.dump(metrics, open(OUT_DIR / f'metrics_{RUN_NAME}.json', 'w'), indent=2)
    print(json.dumps(metrics, indent=2))

In [ ]:
plot_history(history.history, FIG_DIR / f'{RUN_NAME}_curves.png')

**Read the curves (write this in your paper):**
- Train accuracy far above validation accuracy means **overfitting**. Try more dropout, stronger augmentation, or fewer filters.
- Both low and flat means **underfitting**. Try more epochs or a bigger model.
- Validation loss going up while train loss goes down: early stopping should have caught it.

## 6. Validation results

In [ ]:
rep, cm = evaluate_split(model, val_ds, 'validation', FIG_DIR / f'{RUN_NAME}_val_confusion.png')
save_run(history.history, rep, {'epochs_run': len(history.history['loss']),
                                'params': int(model.count_params()),
                                'learning_rate': LEARNING_RATE, 'batch_size': BATCH_SIZE})

## 7. What to do next
1. Record this run in `docs/EXPERIMENT_LOG.md` (settings and val accuracy / macro F1).
2. Download the figures from `road_project/outputs/figures/` and commit them to `docs/figures/` on GitHub.
3. Run notebook 03 (transfer learning) and compare.
4. The trained model is saved at `road_project/models/<RUN_NAME>.keras`. Do **not** commit it to GitHub.